# 04 · CV Pipeline — Detection + Tracking — self-contained

Same as `notebooks/04_detection_pipeline.ipynb` but **all `src` code is inlined**
(no `import src`) — runs standalone on any machine (Windows included). Detectors:
flower (1 class) + insect (`bee, fly, beetle, bug, butterfly`); video does flower
box+ID, insect box+ID+type (BoT-SORT), and distinct per-flower/per-type counts.

The **last section runs inference from trained weights only** — teammates can test
without training or datasets.

## Inlined source

In [ ]:
import os, sys, json, glob
from pathlib import Path
import pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
try:
    import cv2, torch
except Exception:
    pass
_REPO = Path.cwd()
for _c in [_REPO, *_REPO.parents]:
    if (_c / "data").exists() and (_c / "src").exists(): _REPO = _c; break
os.chdir(_REPO)
# ---- src/config.py (inlined; edit here) ----
"""Central configuration for the Bee-A-Hero data pipeline.

All paths are anchored to the repository root (this file lives in ``src/``),
so the pipeline is portable across machines and does not depend on any
absolute/Windows path. Import this module rather than hard-coding paths.
"""

from pathlib import Path

# --- Repository layout -------------------------------------------------------
REPO_ROOT: Path = _REPO

DATA_DIR: Path = REPO_ROOT / "data"
RAW_DIR: Path = DATA_DIR / "raw"
INTERIM_DIR: Path = DATA_DIR / "interim"
PROCESSED_DIR: Path = DATA_DIR / "processed"
BACKUP_DIR: Path = DATA_DIR / "_backup"

INAT_DIR: Path = RAW_DIR / "iNaturist"
FLOWER_DIR: Path = RAW_DIR / "Flower"
# Roboflow bee-detection COCO export (BEE.v8i). Place it here on any machine
# (git-ignored); overridable via the BEE_COCO_DIR env var for other layouts.
BEE_COCO_DIR: Path = Path(__import__("os").environ.get("BEE_COCO_DIR", RAW_DIR / "BEE_coco"))
# Videos to run the visit-counter on.
TEST_VIDEO_DIR: Path = RAW_DIR / "Test_Video"

# Published trained-weights repo on the Hugging Face Hub (for teammates to pull).
HF_WEIGHTS_REPO: str = "Manheim/bee-a-hero-cv"

# iNaturalist splits present on disk. ``public_test`` is unlabeled
# (annotations: 0) and is inference-only — it is NOT part of the labeled split.
INAT_LABELED_SPLITS: tuple[str, ...] = ("train_mini", "val")
INAT_UNLABELED_SPLIT: str = "public_test"

# Generated-artifact locations (git-ignored; see .gitignore data/interim/*).
MANIFEST_DIR: Path = INTERIM_DIR / "manifests"
EDA_DIR: Path = INTERIM_DIR / "eda"
REMOVED_DIR: Path = BACKUP_DIR / "removed"

# --- Reproducibility ---------------------------------------------------------
SEED: int = 42

# --- Split configuration -----------------------------------------------------
# Train/val/test proportions carved from the pooled labeled Insecta images.
SPLIT_RATIOS: dict[str, float] = {"train": 0.70, "val": 0.15, "test": 0.15}

# --- Taxonomy targets --------------------------------------------------------
# Keep only folders whose taxonomic Class == Insecta.
TARGET_CLASS: str = "Insecta"

# Bee families (Hymenoptera) tagged as a subset of interest for the project.
BEE_FAMILIES: frozenset[str] = frozenset({
    "Andrenidae", "Apidae", "Colletidae", "Halictidae",
    "Megachilidae", "Melittidae", "Stenotritidae",
})

# Valid image extensions.
IMAGE_EXTS: frozenset[str] = frozenset({".jpg", ".jpeg", ".png"})

from types import SimpleNamespace
C = SimpleNamespace(**{k: v for k, v in dict(globals()).items() if k.isupper()})
REPO = _REPO
RUNS = C.INTERIM_DIR / "cv_runs"
def exists(p): return Path(p).exists()
print("repo:", REPO, "| device:", "cuda" if ("torch" in dir() and torch.cuda.is_available()) else "cpu")

#### `src/data_pipeline/flower/make_detection_dataset.py` (inlined)

In [ ]:
# ===== src/data_pipeline/flower/make_detection_dataset.py  (inlined module — edit here) =====
import types as _t, sys as _s
mdd = _t.ModuleType("mdd")
mdd.C = C
mdd.__file__ = str(_REPO / "src" / "data_pipeline/flower/make_detection_dataset.py")
_src = r'''
"""
make_detection_dataset.py
-------------------------
Turn the merged Flower Classification dataset (class subfolders) into an
OBJECT-DETECTION-ready dataset in YOLO format, AND emit a labels.csv that
maps every image to its class.

IMPORTANT / HONEST NOTE
-----------------------
The source is a *classification* dataset: one centered flower per image, with
NO ground-truth boxes. This script GENERATES boxes automatically with OpenCV
GrabCut foreground segmentation (a tight box around the detected flower region).
These boxes are approximate and machine-made, not human-verified. They give you
"clear boundaries" to train/prototype a detector, but you should spot-check and
hand-correct a sample before trusting them as ground truth.

INPUT  (created by merge_flowers.py):
    merged_dataset/
        Training Data/<Class>/*.jpeg
        Validation Data/<Class>/*.jpeg
        Testing Data/<Class>/*.jpeg

OUTPUT:
    yolo_dataset/
        images/{train,val,test}/<Class>_<orig>.jpeg
        labels/{train,val,test}/<Class>_<orig>.txt     # "<cls> cx cy w h" (normalized)
        data.yaml
        classes.txt
    labels.csv         # image -> class mapping (classification)
    annotations.csv    # image -> class + pixel bbox (detection, human-readable)

Usage:
    python make_detection_dataset.py
    python make_detection_dataset.py --src merged_dataset --out yolo_dataset --workers 16
    python make_detection_dataset.py --limit 20    # quick smoke test (20 imgs/class)
"""

import argparse
import csv
import os
import shutil
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

import cv2
import numpy as np

IMAGE_EXTS = {".jpeg", ".jpg", ".png"}
SPLIT_MAP = {
    "Training Data": "train",
    "Validation Data": "val",
    "Testing Data": "test",
}
GC_SIZE = 160          # work resolution for GrabCut (speed)
GC_ITERS = 3           # GrabCut iterations
MIN_AREA_FRAC = 0.02   # if fg smaller than this -> fallback box
MAX_AREA_FRAC = 0.985  # if fg bigger than this  -> fallback box
FALLBACK_FRAC = 0.92   # centered fallback box size (fraction of image)


def auto_bbox(img):
    """Return (cx,cy,w,h) normalized [0,1] for the main flower region.
    Falls back to a centered box if segmentation is unreliable."""
    h, w = img.shape[:2]
    scale = GC_SIZE / max(h, w)
    sw, sh = max(1, int(w * scale)), max(1, int(h * scale))
    small = cv2.resize(img, (sw, sh))

    mask = np.zeros((sh, sw), np.uint8)
    bgd = np.zeros((1, 65), np.float64)
    fgd = np.zeros((1, 65), np.float64)
    m = 0.06  # border margin -> assumed background
    rect = (int(sw * m), int(sh * m), int(sw * (1 - 2 * m)), int(sh * (1 - 2 * m)))

    used_fallback = False
    try:
        cv2.grabCut(small, mask, rect, bgd, fgd, GC_ITERS, cv2.GC_INIT_WITH_RECT)
        fg = np.where((mask == cv2.GC_FGD) | (mask == cv2.GC_PR_FGD), 255, 0).astype("uint8")
        fg = cv2.morphologyEx(fg, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8))
        cnts, _ = cv2.findContours(fg, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if cnts:
            c = max(cnts, key=cv2.contourArea)
            x, y, bw, bh = cv2.boundingRect(c)
            area_frac = (bw * bh) / float(sw * sh)
            if not (MIN_AREA_FRAC <= area_frac <= MAX_AREA_FRAC):
                used_fallback = True
        else:
            used_fallback = True
    except Exception:
        used_fallback = True

    if used_fallback:
        f = FALLBACK_FRAC
        cx, cy, nw, nh = 0.5, 0.5, f, f
    else:
        cx = (x + bw / 2) / sw
        cy = (y + bh / 2) / sh
        nw = bw / sw
        nh = bh / sh

    # clamp
    cx, cy = min(max(cx, 0), 1), min(max(cy, 0), 1)
    nw, nh = min(nw, 1), min(nh, 1)
    return cx, cy, nw, nh, used_fallback


def process_one(task):
    """Worker: compute bbox for a single image. Returns dict or None on read error."""
    src, split, cls, cls_id, dst_name = task
    img = cv2.imread(src)
    if img is None:
        return None
    h, w = img.shape[:2]
    cx, cy, nw, nh, fb = auto_bbox(img)
    # pixel coords for the human-readable annotations.csv
    bw_px, bh_px = nw * w, nh * h
    xmin = int(round(cx * w - bw_px / 2))
    ymin = int(round(cy * h - bh_px / 2))
    xmax = int(round(cx * w + bw_px / 2))
    ymax = int(round(cy * h + bh_px / 2))
    xmin, ymin = max(0, xmin), max(0, ymin)
    xmax, ymax = min(w, xmax), min(h, ymax)
    return {
        "src": src, "split": split, "cls": cls, "cls_id": cls_id,
        "dst_name": dst_name, "w": w, "h": h,
        "cx": cx, "cy": cy, "nw": nw, "nh": nh,
        "xmin": xmin, "ymin": ymin, "xmax": xmax, "ymax": ymax,
        "fallback": fb,
    }


def build_tasks(src_root, limit):
    classes = set()
    tasks = []
    per_class_counter = {}
    for split_dir, split_key in SPLIT_MAP.items():
        sdir = src_root / split_dir
        if not sdir.is_dir():
            continue
        for cls_dir in sorted(p for p in sdir.iterdir() if p.is_dir()):
            cls = cls_dir.name
            classes.add(cls)
    class_list = sorted(classes)
    class_id = {c: i for i, c in enumerate(class_list)}

    seen_names = set()
    for split_dir, split_key in SPLIT_MAP.items():
        sdir = src_root / split_dir
        if not sdir.is_dir():
            continue
        for cls_dir in sorted(p for p in sdir.iterdir() if p.is_dir()):
            cls = cls_dir.name
            key = (split_key, cls)
            per_class_counter.setdefault(key, 0)
            for f in sorted(cls_dir.iterdir()):
                if f.suffix.lower() not in IMAGE_EXTS:
                    continue
                if limit and per_class_counter[key] >= limit:
                    break
                per_class_counter[key] += 1
                # unique, collision-proof destination name (flat per split).
                # Dedup on the STEM (not full name) so that e.g. "X.jpeg" and
                # "X.png" don't collide onto the same label .txt file.
                base = f"{cls}_{f.name}"
                stem = Path(base).stem
                dst = base
                i = 1
                while (split_key, Path(dst).stem) in seen_names:
                    dst = f"{stem}_{i}{f.suffix}"
                    i += 1
                seen_names.add((split_key, Path(dst).stem))
                tasks.append((str(f), split_key, cls, class_id[cls], dst))
    return tasks, class_list, class_id


# repo root = .../Bee-A-Hero  (this file is at src/data_pipeline/flower/make_detection_dataset.py)
REPO = Path(__file__).resolve().parents[3]
DEFAULT_SRC = REPO / "data" / "processed" / "flower" / "classification"
DEFAULT_OUT = REPO / "data" / "processed" / "flower" / "yolo"


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--src", default=str(DEFAULT_SRC),
                    help="classification dataset (default: data/processed/flower/classification)")
    ap.add_argument("--out", default=str(DEFAULT_OUT),
                    help="YOLO output (default: data/processed/flower/yolo)")
    ap.add_argument("--workers", type=int, default=max(1, (os.cpu_count() or 4) - 2))
    ap.add_argument("--limit", type=int, default=0, help="max images per class+split (0 = all)")
    args = ap.parse_args()

    src_root = Path(args.src)
    out = Path(args.out)
    for split in ("train", "val", "test"):
        (out / "images" / split).mkdir(parents=True, exist_ok=True)
        (out / "labels" / split).mkdir(parents=True, exist_ok=True)

    tasks, class_list, class_id = build_tasks(src_root, args.limit)
    print(f"Images to process: {len(tasks)}  |  classes: {len(class_list)}  |  workers: {args.workers}")

    labels_rows = []       # image_path, split, class, class_id
    ann_rows = []          # image_path, width, height, class, class_id, xmin, ymin, xmax, ymax
    fallback_count = 0
    done = 0

    with ProcessPoolExecutor(max_workers=args.workers) as ex:
        futs = [ex.submit(process_one, t) for t in tasks]
        for fut in as_completed(futs):
            r = fut.result()
            done += 1
            if r is None:
                continue
            if r["fallback"]:
                fallback_count += 1

            split = r["split"]
            dst_img = out / "images" / split / r["dst_name"]
            dst_lbl = out / "labels" / split / (Path(r["dst_name"]).stem + ".txt")

            shutil.copy2(r["src"], dst_img)
            with open(dst_lbl, "w") as fh:
                fh.write(f"{r['cls_id']} {r['cx']:.6f} {r['cy']:.6f} {r['nw']:.6f} {r['nh']:.6f}\n")

            rel = f"images/{split}/{r['dst_name']}"
            labels_rows.append([rel, split, r["cls"], r["cls_id"]])
            ann_rows.append([rel, r["w"], r["h"], r["cls"], r["cls_id"],
                             r["xmin"], r["ymin"], r["xmax"], r["ymax"]])

            if done % 2000 == 0:
                print(f"  ...{done}/{len(tasks)}")

    # sort for stable output
    labels_rows.sort()
    ann_rows.sort()

    with open(out / "labels.csv", "w", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(["image", "split", "class", "class_id"])
        w.writerows(labels_rows)

    with open(out / "annotations.csv", "w", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(["image", "width", "height", "class", "class_id",
                    "xmin", "ymin", "xmax", "ymax"])
        w.writerows(ann_rows)

    with open(out / "classes.txt", "w") as fh:
        fh.write("\n".join(class_list) + "\n")

    with open(out / "data.yaml", "w") as fh:
        fh.write(f"path: {out.resolve()}\n")
        fh.write("train: images/train\n")
        fh.write("val: images/val\n")
        fh.write("test: images/test\n")
        fh.write(f"nc: {len(class_list)}\n")
        fh.write("names:\n")
        for c in class_list:
            fh.write(f"  - {c}\n")

    print("\n" + "=" * 60)
    print("DETECTION DATASET READY ->", out.resolve())
    print("=" * 60)
    print(f"images written : {len(labels_rows)}")
    print(f"fallback boxes : {fallback_count} "
          f"({100*fallback_count/max(1,len(labels_rows)):.1f}% used centered box)")
    print(f"classes        : {class_list}")
    print("files          : data.yaml, classes.txt, labels.csv, annotations.csv")


if __name__ == "__main__":
    main()

'''
exec(compile(_src, "src/data_pipeline/flower/make_detection_dataset.py", "exec"), mdd.__dict__)
_s.modules["mdd"] = mdd
globals()["mdd"] = mdd

#### `src/cv_engine/prepare_detect.py` (inlined)

In [ ]:
# ===== src/cv_engine/prepare_detect.py  (inlined module — edit here) =====
import types as _t, sys as _s
prepare_detect = _t.ModuleType("prepare_detect")
prepare_detect.C = C
prepare_detect.__file__ = str(_REPO / "src" / "cv_engine/prepare_detect.py")
prepare_detect.auto_bbox = mdd.auto_bbox
_src = r'''
"""Build YOLO **detection** datasets (bbox, no segmentation) from the real,
video-domain datasets the user downloaded into ``data/raw/``.

Two datasets are produced (git-ignored, under ``data/interim/``):

* ``flower_det2``  — single class ``flower``. Sources: the Roboflow flower COCO
  exports + the per-frame flower ROI boxes from the "flower visits" (Ștefan et al.
  2025, time-lapse pollinator) dataset.
* ``insect_multidet`` — multi-class ``[bee, fly, beetle, bug, butterfly]``.
  Sources: "flower visits" full-frame arthropod boxes (taxonomic *order* → coarse
  type) + the Roboflow honey-bee COCO exports (all → ``bee``). Types are learned
  as detection classes, so no fragile separate species classifier is needed.

"flower visits" is time-lapse (many near-identical frames per plant), so splits are
grouped by ``plant_folder`` to avoid train/val leakage.

CLI:  python -m src.cv_engine.prepare_detect            # both
      python -m src.cv_engine.prepare_detect flower     # just flower
      python -m src.cv_engine.prepare_detect insect     # just insect
"""

import json
import random
from collections import Counter, defaultdict
from pathlib import Path

from PIL import Image


RAW = C.RAW_DIR
FV = RAW / "flower visits"                       # Zenodo time-lapse pollinator set
FV_ANN = FV / "annotations" / "raw" / "annotations_full_frames.txt"

# Roboflow COCO exports (split dirs: train/valid/test, _annotations.coco.json)
FLOWER_COCO = ["flowerDetecter-v2.v1i.coco", "flower detection.v3i.coco"]
BEE_COCO = ["Bees detection model.v1i.coco", "Honey Bee Project.v1i.coco",
            "Bee Detection in the Wild/archive", "BEE_coco"]
# Honey Bee Detection Model.v4 is hive-monitoring (dense caged bees) — excluded:
# that domain hurt garden-video mAP in earlier experiments.

_RF_SPLIT = {"train": "train", "valid": "val", "test": "test"}
# coco category NAMES that are NOT a flower (supercategory placeholders / plant parts)
_NOT_FLOWER = {"flowers", "yolov5", "flower-detection", "anther"}
# taxonomic order -> coarse insect class
ORDER2CLS = {"hymenoptera": "bee", "diptera": "fly", "coleoptera": "beetle",
             "hemiptera": "bug", "lepidoptera": "butterfly"}
INSECT_NAMES = ["bee", "fly", "beetle", "bug", "butterfly"]
# per-class cap on flower-visits frames (balance; bee is hugely over-represented)
FV_INSECT_CAP = {"bee": 3000, "fly": 6000, "beetle": 3300, "bug": 900, "butterfly": 200}
FV_FLOWER_CAP = 2500
BEE_COCO_CAP = 3500


# --------------------------------------------------------------------------- #
# COCO (Roboflow) -> YOLO
# --------------------------------------------------------------------------- #
def _coco_split(ds_dir: Path, rf_split: str):
    """Yield (image_path, W, H, [(cls_name, x, y, w, h), ...]) for one split."""
    ann = ds_dir / rf_split / "_annotations.coco.json"
    if not ann.exists():
        return
    coco = json.load(open(ann))
    name = {c["id"]: c["name"].lower() for c in coco["categories"]}
    imgs = {im["id"]: im for im in coco["images"]}
    boxes = defaultdict(list)
    for a in coco["annotations"]:
        boxes[a["image_id"]].append((name.get(a["category_id"], "?"), *a["bbox"]))
    for iid, im in imgs.items():
        p = ds_dir / rf_split / im["file_name"]
        if p.exists():
            yield p, im["width"], im["height"], boxes.get(iid, [])


def _write(out: Path, split: str, img: Path, W, H, lines: list[str]):
    (out / "images" / split).mkdir(parents=True, exist_ok=True)
    (out / "labels" / split).mkdir(parents=True, exist_ok=True)
    link = out / "images" / split / img.name
    if not link.exists():
        try:
            link.symlink_to(img.resolve())
        except FileExistsError:
            pass
    (out / "labels" / split / f"{img.stem}.txt").write_text("\n".join(lines) + ("\n" if lines else ""))


def _yolo_line(cls: int, x, y, w, h, W, H):
    cx, cy = (x + w / 2) / W, (y + h / 2) / H
    return f"{cls} {cx:.6f} {cy:.6f} {w / W:.6f} {h / H:.6f}"


# --------------------------------------------------------------------------- #
# flower visits txt -> rows
# --------------------------------------------------------------------------- #
def _load_fv_rows():
    import csv as _csv
    rows = list(_csv.DictReader(open(FV_ANN), delimiter="\t"))
    return rows


def _fv_group_split(files: list[str], groups: dict[str, str], rng):
    """Assign each plant_folder group to train/val/test 80/10/10 (grouped)."""
    gset = sorted(set(groups[f] for f in files))
    rng.shuffle(gset)
    n = len(gset)
    val = set(gset[: max(1, n // 10)])
    test = set(gset[n // 10: n // 10 + max(1, n // 10)])
    def which(f):
        g = groups[f]
        return "val" if g in val else "test" if g in test else "train"
    return which


# --------------------------------------------------------------------------- #
# Flower dataset
# --------------------------------------------------------------------------- #
def build_flower() -> dict:
    out = C.INTERIM_DIR / "flower_det2"
    stats: Counter = Counter()
    for ds in FLOWER_COCO:
        d = RAW / ds
        for rf, our in _RF_SPLIT.items():
            for p, W, H, boxes in _coco_split(d, rf):
                lines = [_yolo_line(0, x, y, w, h, W, H)
                         for nm, x, y, w, h in boxes if nm not in _NOT_FLOWER and w > 1 and h > 1]
                if lines:
                    _write(out, our, p, W, H, lines); stats[f"{our}_coco"] += 1

    # flower-visits ROI (one flower box per frame), grouped-split, capped
    rows = _load_fv_rows()
    by_file = {}
    groups = {}
    for r in rows:
        f = r["filename_full_frame"]; groups[f] = r["plant_folder"]
        by_file.setdefault(f, r)   # ROI identical across boxes of a frame
    files = list(by_file)
    rng = random.Random(C.SEED); rng.shuffle(files); files = files[:FV_FLOWER_CAP * 2]
    which = _fv_group_split(files, groups, rng)
    kept = Counter()
    for f in files:
        our = which(f)
        if kept[our] >= FV_FLOWER_CAP:
            continue
        r = by_file[f]
        img = FV / "raw" / f
        if not img.exists():
            continue
        try:
            W, H = Image.open(img).size
        except Exception:
            continue
        try:
            x, y, w, h = float(r["x_roi"]), float(r["y_roi"]), float(r["width_roi"]), float(r["height_roi"])
        except ValueError:
            continue
        _write(out, our, img, W, H, [_yolo_line(0, x, y, w, h, W, H)])
        stats[f"{our}_fv"] += 1; kept[our] += 1

    (out / "data.yaml").write_text(
        f"path: {out.resolve()}\ntrain: images/train\nval: images/val\ntest: images/test\n"
        f"nc: 1\nnames: [flower]\n")
    return {"out": str(out), "counts": dict(stats)}


# --------------------------------------------------------------------------- #
# Insect multi-class dataset
# --------------------------------------------------------------------------- #
def build_insect() -> dict:
    out = C.INTERIM_DIR / "insect_multidet"
    cid = {n: i for i, n in enumerate(INSECT_NAMES)}
    stats: Counter = Counter()

    # ---- flower-visits full-frame boxes (order -> class), grouped-split, capped
    rows = _load_fv_rows()
    groups = {r["filename_full_frame"]: r["plant_folder"] for r in rows}
    boxes_by_file = defaultdict(list)
    for r in rows:
        cls = ORDER2CLS.get((r["order"] or "").lower())
        if cls is None:
            continue
        boxes_by_file[r["filename_full_frame"]].append((cls, r))
    files = list(boxes_by_file)
    rng = random.Random(C.SEED); rng.shuffle(files)
    which = _fv_group_split(files, groups, rng)
    cap = Counter()
    for f in files:
        # primary class = first box's class (for capping/balance)
        prim = boxes_by_file[f][0][0]
        if cap[prim] >= FV_INSECT_CAP.get(prim, 3000):
            continue
        img = FV / "raw" / f
        if not img.exists():
            continue
        try:
            W, H = Image.open(img).size
        except Exception:
            continue
        lines = []
        for cls, r in boxes_by_file[f]:
            try:
                x, y, w, h = float(r["x"]), float(r["y"]), float(r["width"]), float(r["height"])
            except ValueError:
                continue
            if w > 1 and h > 1:
                lines.append(_yolo_line(cid[cls], x, y, w, h, W, H))
        if lines:
            _write(out, which(f), img, W, H, lines)
            stats[f"{which(f)}_fv_{prim}"] += 1; cap[prim] += 1

    # ---- bee COCO sets -> all boxes = bee
    for ds in BEE_COCO:
        d = RAW / ds
        kept = 0
        for rf, our in _RF_SPLIT.items():
            for p, W, H, bxs in _coco_split(d, rf):
                if our == "train" and kept >= BEE_COCO_CAP:
                    break
                lines = [_yolo_line(cid["bee"], x, y, w, h, W, H)
                         for nm, x, y, w, h in bxs if w > 1 and h > 1]
                if lines:
                    _write(out, our, p, W, H, lines)
                    stats[f"{our}_bee_{Path(ds).name[:10]}"] += 1
                    if our == "train":
                        kept += 1

    names = ", ".join(INSECT_NAMES)
    (out / "data.yaml").write_text(
        f"path: {out.resolve()}\ntrain: images/train\nval: images/val\ntest: images/test\n"
        f"nc: {len(INSECT_NAMES)}\nnames: [{names}]\n")
    return {"out": str(out), "counts": dict(stats)}


def aug_insect_from_inat(cap_per_class: int = 1500) -> dict:
    """Boost the rare insect classes with iNaturalist crops (GrabCut boxes).

    ``flower visits`` has very few butterflies/bugs/beetles, so top them up from
    iNat: for each taxonomic order we care about, take up to ``cap_per_class``
    images, box the centred organism with GrabCut and add it to ``insect_multidet``.
    """
    import csv as _csv
    import cv2
    out = C.INTERIM_DIR / "insect_multidet"
    cid = {n: i for i, n in enumerate(INSECT_NAMES)}
    order_cls = {"Lepidoptera": "butterfly", "Coleoptera": "beetle", "Hemiptera": "bug"}
    by_cls = defaultdict(list)
    for r in _csv.DictReader(open(C.MANIFEST_DIR / "split_manifest.csv")):
        cl = order_cls.get(r["order"])
        if cl:
            by_cls[cl].append(r)
    rng = random.Random(C.SEED)
    stats: Counter = Counter()
    for cl, recs in by_cls.items():
        rng.shuffle(recs); recs = recs[:cap_per_class]
        for r in recs:
            split = {"train": "train", "val": "val", "test": "test"}.get(r["split"], "train")
            img = C.REPO_ROOT / r["path"]
            im = cv2.imread(str(img))
            if im is None:
                continue
            cx, cy, nw, nh, _ = auto_bbox(im)
            name = f"inat_{cl}_{Path(r['path']).stem}.jpg"
            (out / "images" / split).mkdir(parents=True, exist_ok=True)
            (out / "labels" / split).mkdir(parents=True, exist_ok=True)
            link = out / "images" / split / name
            if not link.exists():
                try:
                    link.symlink_to(img.resolve())
                except FileExistsError:
                    pass
            (out / "labels" / split / f"{Path(name).stem}.txt").write_text(
                f"{cid[cl]} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}\n")
            stats[f"{split}_{cl}"] += 1
    return dict(stats)


if __name__ == "__main__":
    import sys
    what = sys.argv[1] if len(sys.argv) > 1 else "both"
    if what in ("both", "flower"):
        print("FLOWER:", json.dumps(build_flower(), indent=2))
    if what in ("both", "insect"):
        print("INSECT:", json.dumps(build_insect(), indent=2))
    if what in ("both", "insect", "aug"):
        print("AUG:", json.dumps(aug_insect_from_inat(), indent=2))

'''
exec(compile(_src, "src/cv_engine/prepare_detect.py", "exec"), prepare_detect.__dict__)
_s.modules["prepare_detect"] = prepare_detect
globals()["prepare_detect"] = prepare_detect

#### `src/cv_engine/train.py` (inlined)

In [ ]:
# ===== src/cv_engine/train.py  (inlined module — edit here) =====
import types as _t, sys as _s
cvtrain = _t.ModuleType("cvtrain")
cvtrain.C = C
cvtrain.__file__ = str(_REPO / "src" / "cv_engine/train.py")
_src = r'''
"""Fine-tune a YOLO26 detector (flowers or insects) on the RTX 3050 6GB.

Thin, reusable wrapper over Ultralytics. Runs are written under
``data/interim/cv_runs/<name>/`` (git-ignored); best weights at
``.../weights/best.pt``.

CLI:
    python -m src.cv_engine.train --data data/interim/flower_det/data.yaml \
        --name flower_yolo26n --epochs 60 --batch 16
"""

import argparse
import multiprocessing
from pathlib import Path

# Python 3.14 defaults to the "forkserver" start method, which makes Ultralytics'
# DataLoader workers re-import this module and spawn a runaway process swarm with
# no real training. Force "fork" so workers inherit state cleanly.
try:
    multiprocessing.set_start_method("fork", force=True)
except (RuntimeError, ValueError):
    pass  # fork unavailable (Windows) -> use the default start method

from ultralytics import YOLO


RUNS_DIR = C.INTERIM_DIR / "cv_runs"
WEIGHTS_DIR = C.INTERIM_DIR / "weights"


def train(data: str, name: str, model: str = "yolo26n.pt", epochs: int = 60,
          imgsz: int = 640, batch: int = 16, device: int | str = 0,
          patience: int = 15, resume: bool = False, scale: float = 0.5) -> dict:
    RUNS_DIR.mkdir(parents=True, exist_ok=True)
    # prefer a local pretrained copy if present (avoids re-download to cwd)
    local = WEIGHTS_DIR / model
    yolo = YOLO(str(local) if local.exists() else model)
    results = yolo.train(
        data=data, epochs=epochs, imgsz=imgsz, batch=batch, device=device,
        project=str(RUNS_DIR), name=name, seed=C.SEED, deterministic=True,
        amp=True, patience=patience, exist_ok=True, resume=resume, verbose=True,
        # scale jitter: larger range makes the detector see objects at more
        # scales (incl. small) -> tighter boxes on small/video insects.
        scale=scale,
    )
    best = RUNS_DIR / name / "weights" / "best.pt"
    # report validation mAP
    metrics = yolo.val(data=data, imgsz=imgsz, device=device, verbose=False)
    out = {
        "name": name, "best_weights": str(best),
        "map50": round(float(metrics.box.map50), 4),
        "map50_95": round(float(metrics.box.map), 4),
    }
    return out


def main() -> None:
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--data", required=True, help="path to data.yaml")
    ap.add_argument("--name", required=True, help="run name")
    ap.add_argument("--model", default="yolo26n.pt")
    ap.add_argument("--epochs", type=int, default=60)
    ap.add_argument("--imgsz", type=int, default=640)
    ap.add_argument("--batch", type=int, default=16)
    ap.add_argument("--patience", type=int, default=15)
    ap.add_argument("--resume", action="store_true")
    ap.add_argument("--scale", type=float, default=0.5, help="scale-jitter gain (aug)")
    args = ap.parse_args()
    import json
    print(json.dumps(train(args.data, args.name, args.model, args.epochs,
                           args.imgsz, args.batch, patience=args.patience,
                           resume=args.resume, scale=args.scale), indent=2))


if __name__ == "__main__":
    main()

'''
exec(compile(_src, "src/cv_engine/train.py", "exec"), cvtrain.__dict__)
_s.modules["cvtrain"] = cvtrain
globals()["cvtrain"] = cvtrain

#### `src/cv_engine/visit_counter.py` (inlined)

In [ ]:
# ===== src/cv_engine/visit_counter.py  (inlined module — edit here) =====
import types as _t, sys as _s
visit_counter = _t.ModuleType("visit_counter")
visit_counter.C = C
visit_counter.__file__ = str(_REPO / "src" / "cv_engine/visit_counter.py")
_src = r'''
"""Flower-visit counting on video (two-stage insect recognition).

Pipeline:
  1. Detect flowers on **every frame** and keep stable IDs across frames via IoU
     association (handles moving camera/flowers); each flower's box is dilated
     into an approach ROI (``flower_1, flower_2, ...``).
  2. Detect and track insects across frames with a single-class YOLO26 detector
     + BoT-SORT (one track ID per insect).
  3. Classify each tracked insect crop with the iNaturalist-pretrained classifier
     as ``pollinator`` or ``non_pollinator`` (majority vote over a track's life).
  4. Count a visit whenever a tracked insect enters a flower ROI (enter-transition,
     debounced so one dwell = one visit and tracker flicker does not double-count).

Outputs:
  * ``<video>_visits.csv``  -> flower_id, total, pollinator, non_pollinator
  * annotated ``<video>_annotated.mp4`` (flowers + tracks + live counts) if --save-video

CLI:
    python -m src.cv_engine.visit_counter --video data/raw/Test_Video/clip.mp4 \
        --flower-weights .../flower/best.pt --insect-weights .../insect/best.pt \
        --classifier-weights .../insect_classifier/best.pt --save-video
"""

import argparse
import csv
from collections import Counter, defaultdict
from pathlib import Path

import cv2
import numpy as np



def _center(b):
    return (b[0] + b[2]) / 2, (b[1] + b[3]) / 2


def _in(box, pt):
    return box[0] <= pt[0] <= box[2] and box[1] <= pt[1] <= box[3]


class Classifier:
    """iNaturalist-pretrained pollinator/non_pollinator classifier (lazy torch)."""

    def __init__(self, weights: str):
        import timm
        import torch
        self.torch = torch
        ckpt = torch.load(weights, map_location="cpu", weights_only=True)
        self.classes = ckpt["classes"]
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = timm.create_model(ckpt["model"], pretrained=False, num_classes=len(self.classes))
        self.model.load_state_dict(ckpt["state_dict"])
        self.model.eval().to(self.device)
        cfg = timm.data.resolve_data_config({}, model=self.model)
        self.tf = timm.data.create_transform(**cfg, is_training=False)

    def predict(self, crop_bgr) -> str:
        from PIL import Image
        if crop_bgr.size == 0:
            return self.classes[0]
        img = Image.fromarray(cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB))
        x = self.tf(img).unsqueeze(0).to(self.device)
        with self.torch.no_grad():
            idx = int(self.model(x).argmax(1).item())
        return self.classes[idx]


def _iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)
    inter = iw * ih
    ua = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / ua if ua > 0 else 0.0


def _detect_flowers_raw(model, frame, conf, dilate=0.15):
    res = model.predict(frame, conf=conf, verbose=False)[0]
    H, W = frame.shape[:2]
    out = []
    for b in res.boxes.xyxy.cpu().numpy():
        x1, y1, x2, y2 = b
        dw, dh = (x2 - x1) * dilate, (y2 - y1) * dilate
        out.append((max(0, x1 - dw), max(0, y1 - dh), min(W, x2 + dw), min(H, y2 + dh)))
    return out


class FlowerTracker:
    """Per-frame flower detection with stable IDs across frames (IoU association).

    Handles dynamic video (moving camera/flowers): each frame the flowers are
    re-detected and matched to existing tracks by IoU so ``flower_1`` stays the
    same flower as it moves. A short ``max_missed`` grace keeps IDs through brief
    misses. ``seen`` records every flower ID ever assigned (for the final report).
    """

    def __init__(self, model, conf, dilate=0.15, iou_thr=0.3, max_missed=30):
        self.model, self.conf, self.dilate = model, conf, dilate
        self.iou_thr, self.max_missed = iou_thr, max_missed
        self.tracks: dict[str, dict] = {}
        self.next_id = 1
        self.seen: set[str] = set()

    def update(self, frame):
        dets = _detect_flowers_raw(self.model, frame, self.conf, self.dilate)
        used = set()
        for fid in list(self.tracks):
            best, bj = self.iou_thr, -1
            for j, d in enumerate(dets):
                if j in used:
                    continue
                iou = _iou(self.tracks[fid]["box"], d)
                if iou >= best:
                    best, bj = iou, j
            if bj >= 0:
                self.tracks[fid] = {"box": dets[bj], "missed": 0}
                used.add(bj)
            else:
                self.tracks[fid]["missed"] += 1
                if self.tracks[fid]["missed"] > self.max_missed:
                    del self.tracks[fid]
        for j, d in enumerate(dets):
            if j not in used:
                fid = f"flower_{self.next_id}"; self.next_id += 1
                self.tracks[fid] = {"box": d, "missed": 0}
                self.seen.add(fid)
        return self.current()

    def current(self):
        """Active flower boxes without re-detecting (reused between detect frames)."""
        return [(fid, t["box"]) for fid, t in self.tracks.items() if t["missed"] == 0]


POLLINATOR_TYPES = {"bee", "butterfly", "wasp", "fly"}  # highlighted + rolled up in CSV


def _load_types():
    """Map species class_id (str) -> coarse insect type from the taxonomy manifest."""
    import csv as _csv
    bee = set(_C.BEE_FAMILIES)
    out = {}
    try:
        for r in _csv.DictReader(open(_C.MANIFEST_DIR / "split_manifest.csv")):
            o, f = r["order"], r["family"]
            if f in bee: t = "bee"
            elif o == "Lepidoptera": t = "butterfly"
            elif f == "Formicidae": t = "ant"
            elif o == "Coleoptera": t = "beetle"
            elif o == "Diptera": t = "fly"
            elif o == "Hymenoptera": t = "wasp"
            elif o == "Hemiptera": t = "bug"
            else: t = (o or "insect").lower()
            out[r["class_id"]] = t
    except FileNotFoundError:
        pass
    return out


def count_visits(video, flower_weights, insect_weights, classifier_weights,
                 out_dir: Path, conf=0.1, debounce=20, save_video=False,
                 flower_interval=5, vid_stride=2) -> dict:
    from ultralytics import YOLO
    out_dir.mkdir(parents=True, exist_ok=True)
    flower_model, insect_model = YOLO(flower_weights), YOLO(insect_weights)
    classifier = Classifier(classifier_weights) if classifier_weights else None

    types = _load_types() if classifier is not None else {}
    visits = defaultdict(Counter)
    track_votes: dict[int, Counter] = defaultdict(Counter)   # track_id -> class votes
    last_flower: dict[int, str | None] = {}
    last_visit_frame: dict[tuple[int, str], int] = {}
    ftracker = FlowerTracker(flower_model, conf)
    writer = None

    stream = insect_model.track(source=video, stream=True, tracker="botsort.yaml",
                                persist=True, conf=conf, verbose=False, vid_stride=vid_stride)
    for fi, res in enumerate(stream):
        frame = res.orig_img
        # re-detect flowers every `flower_interval` frames (they move slowly);
        # reuse the tracked boxes in between -> dynamic but ~interval× faster.
        flowers = ftracker.update(frame) if fi % flower_interval == 0 else ftracker.current()
        if fi == 0 and save_video:
            h, w = frame.shape[:2]
            writer = cv2.VideoWriter(str(out_dir / (Path(video).stem + "_annotated.mp4")),
                                     cv2.VideoWriter_fourcc(*"mp4v"), 25, (w, h))
        boxes = res.boxes
        labels: dict[int, str] = {}
        if boxes is not None and boxes.id is not None:
            ids = boxes.id.int().cpu().tolist()
            xyxy = boxes.xyxy.cpu().numpy()
            for tid, b in zip(ids, xyxy):
                if classifier is not None:                    # majority vote per track
                    x1, y1, x2, y2 = map(int, b)
                    sp = classifier.predict(frame[y1:y2, x1:x2])
                    track_votes[tid][types.get(sp, sp)] += 1   # species id -> type
                    cls = track_votes[tid].most_common(1)[0][0]
                else:
                    cls = "insect"
                labels[tid] = cls
                cur = next((fid for fid, fb in flowers if _in(fb, _center(b))), None)
                if cur is not None and last_flower.get(tid) != cur:
                    if fi - last_visit_frame.get((tid, cur), -10**9) > debounce:
                        visits[cur]["total"] += 1
                        visits[cur][cls] += 1
                        last_visit_frame[(tid, cur)] = fi
                last_flower[tid] = cur
        if writer is not None:
            writer.write(_annotate(frame, flowers, res, labels, visits))
    if writer is not None:
        writer.release()

    for fid in ftracker.seen:                     # include flowers seen with 0 visits
        visits[fid]
    ctypes = sorted({t for v in visits.values() for t in v if t != "total"})
    rows = [{"flower_id": k, "total": v["total"],
             "pollinator": sum(v.get(t, 0) for t in POLLINATOR_TYPES),
             "non_pollinator": v["total"] - sum(v.get(t, 0) for t in POLLINATOR_TYPES),
             **{t: v.get(t, 0) for t in ctypes}}
            for k, v in sorted(visits.items())]
    csv_path = out_dir / (Path(video).stem + "_visits.csv")
    with open(csv_path, "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=["flower_id", "total", "pollinator", "non_pollinator"] + ctypes)
        w.writeheader(); w.writerows(rows)
    return {"video": Path(video).name, "flowers": len(flowers),
            "visits": {r["flower_id"]: r["total"] for r in rows}, "csv": str(csv_path)}


def _annotate(frame, flowers, res, labels, visits):
    for fid, (x1, y1, x2, y2) in flowers:
        cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 200, 0), 2)
        cv2.putText(frame, f"{fid}:{visits[fid]['total']}", (int(x1), int(y1) - 6),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 200, 0), 2)
    if res.boxes is not None and res.boxes.id is not None:
        for tid, b in zip(res.boxes.id.int().cpu().tolist(), res.boxes.xyxy.cpu().numpy()):
            x1, y1, x2, y2 = map(int, b)
            cls = labels.get(tid, "insect")
            poll = cls in POLLINATOR_TYPES
            col = (0, 180, 255) if poll else (0, 0, 255)   # pollinator=orange, else red
            cv2.rectangle(frame, (x1, y1), (x2, y2), col, 2)
            tag = f"{cls}{' (pollinator)' if poll else ''} #{tid}"
            cv2.putText(frame, tag, (x1, y1 - 6),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, col, 1)
    return frame


def main() -> None:
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--video", required=True)
    ap.add_argument("--flower-weights", required=True)
    ap.add_argument("--insect-weights", required=True)
    ap.add_argument("--classifier-weights", default="")
    ap.add_argument("--out", default=str(C.INTERIM_DIR / "cv_runs" / "visits"))
    ap.add_argument("--conf", type=float, default=0.1)
    ap.add_argument("--debounce", type=int, default=20)
    ap.add_argument("--flower-interval", type=int, default=5,
                    help="re-detect flowers every N frames (dynamic video)")
    ap.add_argument("--vid-stride", type=int, default=2,
                    help="process every Nth frame (lower effective fps, stabler tracks)")
    ap.add_argument("--save-video", action="store_true")
    args = ap.parse_args()
    import json
    print(json.dumps(count_visits(args.video, args.flower_weights, args.insect_weights,
                                  args.classifier_weights, Path(args.out), args.conf,
                                  args.debounce, args.save_video, args.flower_interval,
                                  args.vid_stride), indent=2))


if __name__ == "__main__":
    main()

'''
exec(compile(_src, "src/cv_engine/visit_counter.py", "exec"), visit_counter.__dict__)
_s.modules["visit_counter"] = visit_counter
globals()["visit_counter"] = visit_counter

#### `src/cv_engine/video_detect.py` (inlined)

In [ ]:
# ===== src/cv_engine/video_detect.py  (inlined module — edit here) =====
import types as _t, sys as _s
video_detect = _t.ModuleType("video_detect")
video_detect.C = C
video_detect.__file__ = str(_REPO / "src" / "cv_engine/video_detect.py")
for _n in ("FlowerTracker", "_center", "_in"):
    setattr(video_detect, _n, getattr(visit_counter, _n))
_src = r'''
"""Flower-visit counting on video by **detection + tracking** (bbox, no segmentation).

Single real-time camera stream, resampled to a fixed **24 fps**. Per frame:

  1. Flowers: per-frame YOLO detection with stable IDs (``FlowerTracker``) -> a
     **separate** box per flower (never unified).
  2. Insects: multi-class YOLO26 detector (``bee, fly, beetle, bug, butterfly``)
     + BoT-SORT -> one **track ID + type** per insect. Each insect keeps its own
     colour (by track ID) so bee #1 and bee #2 are distinct.
  3. Type: taken directly from the detector, **majority-voted over the track's life**
     for stability (no separate classifier).
  4. Visit: counted when a tracked insect's box centre enters a flower box, each
     ``(insect, flower)`` pair **once ever** (a fly-off + return is not a new visit),
     stamped with a **timestamp**.

Outputs (to ``test_video_result/``):
  * ``<video>_visits.csv``    -> flower_id, total, <per type ...>
  * ``<video>_timeline.csv``  -> flower_id, track_id, type, t_enter_s  (one row / visit)
  * annotated ``<video>_annotated.mp4`` (flower boxes + per-insect boxes/IDs + counts)

CLI:
    python -m src.cv_engine.video_detect --video data/raw/Test_Video/clip.mp4 \
        --flower-weights .../flower/best.pt --insect-weights .../insect/best.pt --save-video
"""

import argparse
import csv
import colorsys
from collections import Counter, defaultdict
from pathlib import Path

import cv2


TARGET_FPS = 24
POLLINATORS = {"bee", "butterfly", "fly"}      # rolled up in the CSV as "pollinator"


def _color(tid: int):
    """Deterministic distinct BGR colour per track ID (golden-ratio hue hop)."""
    h = (tid * 0.61803398875) % 1.0
    r, g, b = colorsys.hsv_to_rgb(h, 0.85, 1.0)
    return int(b * 255), int(g * 255), int(r * 255)


def count_visits_det(video, flower_weights, insect_weights, out_dir: Path,
                     conf=0.25, flower_conf=0.15, save_video=False,
                     flower_interval=5, target_fps=TARGET_FPS) -> dict:
    from ultralytics import YOLO
    out_dir.mkdir(parents=True, exist_ok=True)
    flower_model, insect_model = YOLO(flower_weights), YOLO(insect_weights)
    names = insect_model.names                                 # {cls_id: type}

    in_fps = cv2.VideoCapture(str(video)).get(cv2.CAP_PROP_FPS) or 30.0
    stride = max(1, round(in_fps / target_fps))
    out_fps = in_fps / stride

    visits = defaultdict(Counter)
    timeline: list[dict] = []
    votes: dict[int, Counter] = defaultdict(Counter)          # track_id -> type votes
    counted: set[tuple[int, str]] = set()                     # (track_id, flower_id) once
    ftracker = FlowerTracker(flower_model, flower_conf)
    writer = None

    stream = insect_model.track(source=video, stream=True, tracker="botsort.yaml",
                                persist=True, conf=conf, verbose=False, vid_stride=stride)
    for fi, res in enumerate(stream):
        frame = res.orig_img
        H, W = frame.shape[:2]
        t_s = round(fi * stride / in_fps, 2)
        flowers = ftracker.update(frame) if fi % flower_interval == 0 else ftracker.current()
        if fi == 0 and save_video:
            writer = cv2.VideoWriter(str(out_dir / (Path(video).stem + "_annotated.mp4")),
                                     cv2.VideoWriter_fourcc(*"mp4v"), out_fps, (W, H))
        drawn = []
        b = res.boxes
        if b is not None and b.id is not None:
            ids = b.id.int().cpu().tolist()
            xyxy = b.xyxy.cpu().numpy()
            cls = b.cls.int().cpu().tolist()
            for tid, box, c in zip(ids, xyxy, cls):
                votes[tid][names[c]] += 1
                typ = votes[tid].most_common(1)[0][0]          # stable majority type
                cur = next((fid for fid, fb in flowers if _in(fb, _center(box))), None)
                if cur is not None and (tid, cur) not in counted:
                    counted.add((tid, cur))
                    visits[cur]["total"] += 1
                    visits[cur][typ] += 1
                    timeline.append({"flower_id": cur, "track_id": tid, "type": typ, "t_enter_s": t_s})
                drawn.append((tid, box, typ))
        if writer is not None:
            writer.write(_annotate(frame, flowers, drawn, visits))
    if writer is not None:
        writer.release()

    for fid in ftracker.seen:
        visits[fid]
    ctypes = sorted({t for v in visits.values() for t in v if t != "total"})
    rows = [{"flower_id": k, "total": v["total"],
             "pollinator": sum(v.get(t, 0) for t in POLLINATORS),
             **{t: v.get(t, 0) for t in ctypes}}
            for k, v in sorted(visits.items())]
    csv_path = out_dir / (Path(video).stem + "_visits.csv")
    with open(csv_path, "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=["flower_id", "total", "pollinator"] + ctypes)
        w.writeheader(); w.writerows(rows)
    tl = out_dir / (Path(video).stem + "_timeline.csv")
    with open(tl, "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=["flower_id", "track_id", "type", "t_enter_s"])
        w.writeheader(); w.writerows(timeline)
    return {"video": Path(video).name, "flowers": len(ftracker.seen), "out_fps": round(out_fps, 1),
            "visits": {r["flower_id"]: r["total"] for r in rows}, "csv": str(csv_path), "timeline": str(tl)}


def aggregate_csvs(out_dir: Path) -> dict:
    """Merge every per-video CSV into two team-friendly tables the ML/LLM team can fetch:

      * ``ALL_visits.csv``   -> video, flower_id, total, pollinator, <types...>
      * ``ALL_timeline.csv`` -> video, flower_id, track_id, type, t_enter_s
    """
    import glob
    out_dir = Path(out_dir)
    for kind, key in (("visits", "_visits.csv"), ("timeline", "_timeline.csv")):
        rows, fields = [], []
        for f in sorted(glob.glob(str(out_dir / f"*{key}"))):
            if Path(f).name.startswith("ALL_"):
                continue
            video = Path(f).name[: -len(key)]
            for r in csv.DictReader(open(f)):
                for k in r:
                    if k not in fields:
                        fields.append(k)
                rows.append({"video": video, **r})
        dst = out_dir / f"ALL_{kind}.csv"
        with open(dst, "w", newline="") as fh:
            w = csv.DictWriter(fh, fieldnames=["video"] + fields, restval=0)
            w.writeheader(); w.writerows(rows)
    return {"all_visits": str(out_dir / "ALL_visits.csv"),
            "all_timeline": str(out_dir / "ALL_timeline.csv")}


def _annotate(frame, flowers, drawn, visits):
    for tid, box, typ in drawn:                                # per-insect box + id + type
        x1, y1, x2, y2 = map(int, box)
        col = _color(tid)
        cv2.rectangle(frame, (x1, y1), (x2, y2), col, 2)
        cv2.putText(frame, f"{typ} #{tid}", (x1, max(12, y1 - 6)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, col, 2)
    for fid, (x1, y1, x2, y2) in flowers:                      # separate box per flower
        cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 200, 0), 2)
        cv2.putText(frame, f"{fid}:{visits[fid]['total']}", (int(x1), int(y1) - 6),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 200, 0), 2)
    return frame


def main() -> None:
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--video", required=True)
    ap.add_argument("--flower-weights", required=True)
    ap.add_argument("--insect-weights", required=True, help="multi-class YOLO26 insect detector")
    ap.add_argument("--out", default=str(C.REPO_ROOT / "test_video_result"))
    ap.add_argument("--conf", type=float, default=0.25)
    ap.add_argument("--flower-conf", type=float, default=0.15)
    ap.add_argument("--flower-interval", type=int, default=5)
    ap.add_argument("--target-fps", type=int, default=TARGET_FPS)
    ap.add_argument("--save-video", action="store_true")
    args = ap.parse_args()
    import json
    print(json.dumps(count_visits_det(args.video, args.flower_weights, args.insect_weights,
                                      Path(args.out), args.conf, args.flower_conf,
                                      args.save_video, args.flower_interval, args.target_fps), indent=2))


if __name__ == "__main__":
    main()

'''
exec(compile(_src, "src/cv_engine/video_detect.py", "exec"), video_detect.__dict__)
_s.modules["video_detect"] = video_detect
globals()["video_detect"] = video_detect

## 1 · Build detection datasets (from data/raw)

In [ ]:
flower_yaml = C.INTERIM_DIR / "flower_det2" / "data.yaml"
insect_yaml = C.INTERIM_DIR / "insect_multidet" / "data.yaml"
if not flower_yaml.exists(): print(prepare_detect.build_flower())
if not insect_yaml.exists():
    print(prepare_detect.build_insect()); print(prepare_detect.aug_insect_from_inat())
print("flower:", flower_yaml.exists(), "| insect:", insect_yaml.exists())

## 2 · Train the detectors (skip if weights exist)

In [ ]:
flower_w = RUNS / "flower_det2_yolo26m" / "weights" / "best.pt"
insect_w = RUNS / "insect_multidet_yolo26m" / "weights" / "best.pt"
if not flower_w.exists():
    cvtrain.train(str(flower_yaml), "flower_det2_yolo26m", model="yolo26m.pt", epochs=100, batch=4, patience=20)
if not insect_w.exists():
    cvtrain.train(str(insect_yaml), "insect_multidet_yolo26m", model="yolo26m.pt", epochs=60, batch=4, patience=20)
print("flower:", exists(flower_w), "| insect:", exists(insect_w))

## 3 · Metrics

In [ ]:
def best_map(run):
    df = pd.read_csv(RUNS / run / "results.csv"); df.columns=[c.strip() for c in df.columns]
    i = df["metrics/mAP50-95(B)"].idxmax()
    return {k: round(float(df.loc[i, f"metrics/{k}(B)"]),4) for k in ["mAP50","mAP50-95","precision","recall"]}
metrics = {"flower": best_map("flower_det2_yolo26m"), "insect": best_map("insect_multidet_yolo26m")}
print(json.dumps(metrics, indent=2))

## 4 · Run on the test videos

In [ ]:
out = REPO / "test_video_result"; out.mkdir(exist_ok=True)
results = []
for v in sorted(C.TEST_VIDEO_DIR.glob("*.mp4")):
    r = video_detect.count_visits_det(str(v), str(flower_w), str(insect_w), out, save_video=True)
    results.append(r); print(v.name, "->", r["flowers"], "flowers,", sum(r["visits"].values()), "visits @", r["out_fps"], "fps")
print("team CSVs:", video_detect.aggregate_csvs(out))   # ALL_visits.csv + ALL_timeline.csv

### Visit tallies (per flower, per insect type)

In [ ]:
frames=[]
for f in sorted(out.glob("*_visits.csv")):
    d=pd.read_csv(f); d.insert(0,"video",f.stem.replace("_visits","")); frames.append(d)
vdf=pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
vdf[vdf["total"]>0] if not vdf.empty else vdf

### Visit timeline (timestamps)

In [ ]:
tls=[]
for f in sorted(out.glob("*_timeline.csv")):
    d=pd.read_csv(f)
    if len(d): d.insert(0,"video",f.stem.replace("_timeline","")); tls.append(d)
tdf=pd.concat(tls, ignore_index=True) if tls else pd.DataFrame(); tdf.head(20)

### Sample annotated frame

In [ ]:
vids=sorted(glob.glob(str(out/"*_annotated.mp4")))
if vids:
    cap=cv2.VideoCapture(vids[0]); cap.set(cv2.CAP_PROP_POS_FRAMES,int(cap.get(cv2.CAP_PROP_FRAME_COUNT)*0.5))
    ok,fr=cap.read(); cap.release()
    if ok:
        plt.figure(figsize=(11,6)); plt.axis("off"); plt.title(Path(vids[0]).stem)
        plt.imshow(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)); plt.show()

## 5 · ⚡ ONE-SHOT TEST — metrics + run (weights ship in the repo)
Run the inlined-module cells at the top once, then this cell: it prints the
detector **metrics** and runs the pipeline on a test video (annotated video + team CSVs).

In [ ]:
flower_w = RUNS / "flower_det2_yolo26m" / "weights" / "best.pt"
insect_w = RUNS / "insect_multidet_yolo26m" / "weights" / "best.pt"
assert flower_w.exists() and insect_w.exists(), "weights missing — pull the repo with the committed best.pt files"
def _best(run):
    df = pd.read_csv(RUNS / run / "results.csv"); df.columns = [c.strip() for c in df.columns]
    i = df["metrics/mAP50-95(B)"].idxmax()
    return {k: round(float(df.loc[i, f"metrics/{k}(B)"]), 4) for k in ["mAP50","mAP50-95","precision","recall"]}
print("METRICS:", json.dumps({"flower": _best("flower_det2_yolo26m"), "insect": _best("insect_multidet_yolo26m")}, indent=2))
out = REPO / "test_video_result"; out.mkdir(exist_ok=True)
clip = sorted(C.TEST_VIDEO_DIR.glob("*.mp4"))[0]
r = video_detect.count_visits_det(str(clip), str(flower_w), str(insect_w), out, save_video=True)
print("RUN:", json.dumps(r, indent=2)); print("team CSVs:", video_detect.aggregate_csvs(out))

## 6 · Summary

In [ ]:
print(json.dumps({"metrics": metrics, "videos": len(results),
                  "total_visits": int(sum(sum(r["visits"].values()) for r in results))}, indent=2))